# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [5]:
"""The following generates the Quiz all our models will be evaluated on."""

import string
import yaml

import logging

from dotenv import load_dotenv
from pathlib import Path

logging.basicConfig(level=logging.INFO)
load_dotenv(Path.cwd() / "keys.env", verbose=True)

from smolbench.induction.periodic import (
    PeriodicConfig,
    Prompter,
    get_periodic_numeric_quiz,
    numeric_count_query_gen,
)
from smolbench.evals import Marks
import os
from datetime import timedelta

# SageMaker auth: mint a short-lived (<=12h) bearer token from AWS creds.
# Uses botocore directly to avoid the sagemaker.core import chain (rpds compat issue).
# Set AWS_INFERENCE_API_KEY yourself to skip minting.
if not os.getenv("AWS_INFERENCE_API_KEY"):
    import base64
    from botocore.auth import SigV4QueryAuth
    from botocore.awsrequest import AWSRequest
    from botocore.session import Session as BotocoreSession

    _region = os.getenv("AWS_REGION", "us-east-1")
    _credentials = BotocoreSession().get_credentials()
    if _credentials is None:
        raise RuntimeError("No AWS credentials found. Check your environment or credential provider.")
    _request = AWSRequest(
        method="POST",
        url="https://sagemaker.amazonaws.com/",
        headers={"host": "sagemaker.amazonaws.com"},
        params={"Action": "CallWithBearerToken"},
    )
    SigV4QueryAuth(_credentials, "sagemaker", _region, expires=43200).add_auth(_request)
    _presigned_url = _request.url.replace("https://", "") + "&Version=1"
    os.environ["AWS_INFERENCE_API_KEY"] = (
        "sagemaker-api-key-" + base64.b64encode(_presigned_url.encode()).decode()
    )

# Per-archetype SageMaker ENDPOINT NAMES -- replace with your deployed endpoints.
# The provider builds .../endpoints/<name>/openai/v1 from AWS_INFERENCE_BASE_URL.
DENSE_MODEL = "llama-31-405b"      # Llama-3.1-405B (dense)
COT_MODEL = "nemotron-ultra-253b"  # Nemotron-Ultra-253B (dense CoT)
MOE_MODEL = "llama4-maverick"      # Llama-4-Maverick (MoE)

from smolbench.evals.provider import evaluate

template = string.Template(
    "You are a precise integer counter.\n"
    "\n"
    "Task: answer the question below with a single integer and nothing else.\n"
    "\n"
    "Output format:\n"
    "Return exactly one integer and nothing else.\n"
    "Do not output any explanation, punctuation, quotes, or extra whitespace.\n"
    "Stop immediately after writing the integer.\n"
    "\n"
    "Context:\n"
    "There is a counting game. Positions are counted starting from 1. "
    "At each position, words are written according to the following rules:\n"
    "$positive_info\n"
    "Question:\n"
    "How many of the positions 1 through $seq_len include '$label'?"
)

SEED: int = 1776
intens_quiz, extens_quiz, noise_intens_quiz = get_periodic_numeric_quiz(
    PeriodicConfig(
        n=9,
        labels=9,
        seed=SEED,
    ),
    Prompter(
        template,
        {},
        numeric_count_query_gen,
    ),
)

RuntimeError: No AWS credentials found. Check your environment or credential provider.

In [ ]:
# Endpoint deploy/teardown now lives in smolbench/evals/aws.py.
from smolbench.evals.aws import provision_endpoint

## Prompt Validation

In [ ]:
print(intens_quiz[0].prompt)

In [ ]:
print(extens_quiz[0].prompt)

In [ ]:
print(noise_intens_quiz[0].prompt)

# Decoder-Only Model
This section tests classical decoder-only models.

In [ ]:
# Provisions DENSE_MODEL's endpoint for this section, then tears it down on exit
# (only one endpoint is live at a time). If provisioning fails the eval vars are
# never assigned and the results cell below will NameError -- that is expected.
with provision_endpoint(DENSE_MODEL):
    decode_intens_eval: Marks = evaluate(intens_quiz, DENSE_MODEL, SEED)
    decode_extens_eval: Marks = evaluate(extens_quiz, DENSE_MODEL, SEED)
    decode_noise_intens_eval: Marks = evaluate(noise_intens_quiz, DENSE_MODEL, SEED)

In [ ]:
# Prints out results.
print(decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid)
print(decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid)
print(decode_noise_intens_eval.correct, decode_noise_intens_eval.incorrect, decode_noise_intens_eval.invalid)
# Serializes results.
with open("results/decode_intens.yaml", 'w') as file:
    yaml.dump(
        decode_intens_eval, file, default_flow_style=False, indent=4
    )
with open("results/decode_extens.yaml", 'w') as file:
    yaml.dump(
        decode_extens_eval, file, default_flow_style=False, indent=4
    )
with open("results/decode_noise_intens.yaml", 'w') as file:
    yaml.dump(
        decode_noise_intens_eval, file, default_flow_style=False, indent=4
    )

## CoT Model
This section tests a CoT model.

In [ ]:
# Provisions COT_MODEL's endpoint for this section, then tears it down on exit.
with provision_endpoint(COT_MODEL):
    cot_intens_eval: Marks = evaluate(intens_quiz, COT_MODEL, SEED, extra_args={"max_completion_tokens": 8192})
    cot_extens_eval: Marks = evaluate(extens_quiz, COT_MODEL, SEED, extra_args={"max_completion_tokens": 8192})
    cot_noise_intens_eval: Marks = evaluate(noise_intens_quiz, COT_MODEL, SEED, extra_args={"max_completion_tokens": 8192})

In [ ]:
# Prints out results.
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)
# print(cot_noise_intens_eval.correct, cot_noise_intens_eval.incorrect, cot_noise_intens_eval.invalid)
# Serializes results.
with open("results/cot_intens.yaml", 'w') as file:
    yaml.dump(
        cot_intens_eval, file, default_flow_style=False, indent=4
    )
with open("results/cot_extens.yaml", 'w') as file:
    yaml.dump(
        cot_extens_eval, file, default_flow_style=False, indent=4
    )
with open("results/cot_noise_intens.yaml", 'w') as file:
    yaml.dump(
        cot_noise_intens_eval, file, default_flow_style=False, indent=4
    )

## MoE Model
This section tests an MoE model.

In [ ]:
# Provisions MOE_MODEL's endpoint for this section, then tears it down on exit.
with provision_endpoint(MOE_MODEL):
    moe_intens_eval: Marks = evaluate(intens_quiz, MOE_MODEL, SEED)
    moe_extens_eval: Marks = evaluate(extens_quiz, MOE_MODEL, SEED)
    moe_noise_intens_eval: Marks = evaluate(noise_intens_quiz, MOE_MODEL, SEED)

In [ ]:
# Prints out results.
print(moe_intens_eval.correct, moe_intens_eval.incorrect, moe_intens_eval.invalid)
print(moe_extens_eval.correct, moe_extens_eval.incorrect, moe_extens_eval.invalid)
print(moe_noise_intens_eval.correct, moe_noise_intens_eval.incorrect, moe_noise_intens_eval.invalid)
# Serializes results.
with open("results/moe_intens.yaml", 'w') as file:
    yaml.dump(
        moe_intens_eval, file, default_flow_style=False, indent=4
    )
with open("results/moe_extens.yaml", 'w') as file:
    yaml.dump(
        moe_extens_eval, file, default_flow_style=False, indent=4
    )
with open("results/moe_noise_intens.yaml", 'w') as file:
    yaml.dump(
        moe_noise_intens_eval, file, default_flow_style=False, indent=4
    )

# HRM Model
This section tests an HRM model.

In [ ]:
# hrm_intens_eval: Marks = evaluate(intens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
# hrm_extens_eval: Marks = evaluate(extens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
# print(hrm_intens_eval.correct, hrm_intens_eval.incorrect, hrm_intens_eval.invalid)
# print(hrm_extens_eval.correct, hrm_extens_eval.incorrect, hrm_extens_eval.invalid)
# with open('results/hrm_intens.yaml', 'w') as file:
#     yaml.dump(hrm_intens_eval, file, default_flow_style=False, indent=4)
# with open('results/hrm_extens.yaml', 'w') as file:
#     yaml.dump(hrm_extens_eval, file, default_flow_style=False, indent=4)